# Setup SLM & Ngrok Tunnel trên Kaggle (Bằng Transformers gốc)
Notebook này được thiết kế để chạy Qwen3 trên GPU của Kaggle sử dụng thư viện `transformers` nổi tiếng (ổn định tuyệt đối, không sợ kén phần cứng như `vllm`). Nó bọc mô hình bằng FastAPI và tạo Public URL vượt tường lửa qua Ngrok.

**Lưu ý quan trọng**: Đảm bảo bật **GPU (T4x2)** trên Kaggle.

In [1]:
!pip install -q pyngrok fastapi uvicorn transformers accelerate nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installe

In [ ]:
import os
from pyngrok import ngrok
import nest_asyncio

# LẤY TOKEN NGROK CỦA BẠN TỪ TRANG CHỦ NGROK.COM
NGROK_AUTH_TOKEN = "3Cydo6gmQFjSDQPw4jJbSLOR0fY_5XeBkUMdiwt6wdMnP7Evd"

# Cấp quyền cho tunnel Ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Bật bypass loop cho FastAPI chạy mượt mà trong Jupyter Notebook
nest_asyncio.apply()

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import torch

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

print(f"⏳ Đang tải siêu mô hình {MODEL_NAME} bằng thuật toán cốt lõi Transformers. Chờ xíu nhé...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# device_map="auto" tự động phân tích và gắn model dàn đều lên cả 2 card T4 của Kaggle siêu an toàn!
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
print("✅ Qwen3-4B đã lên nòng (Giao diện Transformers)!")

app = FastAPI()

@app.post("/v1/generate")
async def generate(request: Request):
    data = await request.json()
    prompt = data.get("prompt", "")
    
    # Dùng chat template chuẩn hệ thống của Qwen3 để giữ mạch logic cao nhất
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    try:
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=512,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
        return JSONResponse(content={"text": generated_text})
    finally:
        import gc
        del model_inputs, generated_ids
        gc.collect()
        torch.cuda.empty_cache()

⏳ Đang tải siêu mô hình Qwen/Qwen3-4B-Instruct-2507 bằng thuật toán cốt lõi Transformers. Chờ xíu nhé...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

✅ Qwen3-4B đã lên nòng (Giao diện Transformers)!


In [4]:
# Chặn rác đường hầm trước khi chạy mới
ngrok.kill()

# Đào đường hầm Ngrok tới cổng 8000 của FastAPI
public_url = ngrok.connect(8000).public_url

print("=" * 75)
print(f"🔥 ĐÂY LÀ URL PUBLIC CỦA BẠN: {public_url}")
print("👉 SAO CHÉP URL TRÊN VÀ DÁN VÀO BIẾN SLM_NGROK_URL TRONG FILE .env CỦA BACKEND!")
print("=" * 75)

# Hoạt động web server để hứng requests 
uvicorn.run(app, host="0.0.0.0", port=8000)

🔥 ĐÂY LÀ URL PUBLIC CỦA BẠN: https://opacity-mournful-rejoicing.ngrok-free.dev
👉 SAO CHÉP URL TRÊN VÀ DÁN VÀO BIẾN SLM_NGROK_URL TRONG FILE .env CỦA BACKEND!


INFO:     Started server process [47]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/tmp/ipykernel_47/1116758498.py", line 39, in generate
    generated_ids = model.generate(
                    ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py", line 2625, in generate
    result = self._sample(
             ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py", line 3606, in _sample
    outputs = self(**model_inputs, return_dict=True)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/

INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 500 Internal Server Error


...

INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/tmp/ipykernel_47/1116758498.py", line 39, in generate
    generated_ids = model.generate(
                    ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py", line 2625, in generate
    result = self._sample(
             ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py", line 3606, in _sample
    outputs = self(**model_inputs, return_dict=True)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/

INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63a7:8e7d:b53b:a33c:11c8:1d86:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2402:800:63

INFO:     Shutting down


RuntimeError: Event loop stopped before Future completed.